<a href="https://colab.research.google.com/github/PawanPPatil/stationery-ai-chatbot/blob/main/Gen_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re



In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
df = pd.read_csv("3094_Sales_Deta_d_Itemwise_272820_2026_02_11_122611.csv")

df.columns = df.columns.str.strip()

# Fix date (IMPORTANT)
df['Bill Dt.'] = pd.to_datetime(df['Bill Dt.'], dayfirst=True, errors='coerce')
df = df[df['Bill Dt.'].notna()]

# Date columns
df['Date'] = df['Bill Dt.'].dt.date
df['Year'] = df['Bill Dt.'].dt.year
df['Month'] = df['Bill Dt.'].dt.month

# Clean numeric
df['Sold Qty'] = pd.to_numeric(df['Sold Qty'], errors='coerce').fillna(0)

df['MRP'] = df['MRP'].astype(str).str.replace(',', '').str.replace('₹', '')
df['MRP'] = pd.to_numeric(df['MRP'], errors='coerce').fillna(0)

# Revenue
df['Revenue'] = df['Sold Qty'] * df['MRP']

# Clean text
df['BRAND'] = df['BRAND'].str.lower()
df['CATEGORY'] = df['CATEGORY'].str.lower()
df['Item Name'] = df['Item Name'].str.lower()

In [ ]:
today = df['Date'].max()

unique_dates = sorted(df['Date'].unique())
yesterday = unique_dates[-2] if len(unique_dates) > 1 else today

today_data = df[df['Date'] == today]
yesterday_data = df[df['Date'] == yesterday]

today_rev = today_data['Revenue'].sum()
today_qty = today_data['Sold Qty'].sum()

yesterday_rev = yesterday_data['Revenue'].sum()
yesterday_qty = yesterday_data['Sold Qty'].sum()

In [ ]:
last_7_days = df[df['Date'] >= (today - pd.Timedelta(days=7))]

prev_7_days = df[(df['Date'] < (today - pd.Timedelta(days=7))) &
                 (df['Date'] >= (today - pd.Timedelta(days=14)))]

this_week_rev = last_7_days['Revenue'].sum()
last_week_rev = prev_7_days['Revenue'].sum()

this_week_qty = last_7_days['Sold Qty'].sum()
last_week_qty = prev_7_days['Sold Qty'].sum()

week_growth = ((this_week_rev - last_week_rev) / last_week_rev * 100) if last_week_rev != 0 else 0

In [ ]:
current_month = today.month
current_year = today.year

this_month = df[(df['Month'] == current_month) & (df['Year'] == current_year)]

if current_month == 1:
    last_month = df[(df['Month'] == 12) & (df['Year'] == current_year - 1)]
else:
    last_month = df[(df['Month'] == current_month - 1) & (df['Year'] == current_year)]

this_month_rev = this_month['Revenue'].sum()
last_month_rev = last_month['Revenue'].sum()

month_growth = ((this_month_rev - last_month_rev) / last_month_rev * 100) if last_month_rev != 0 else 0

In [ ]:
brand_sales = df.groupby('BRAND')['Revenue'].sum().sort_values(ascending=False)

item_sales = df.groupby('Item Name')['Revenue'].sum().sort_values(ascending=False)

category_sales = df.groupby('CATEGORY')['Revenue'].sum().sort_values(ascending=False)

In [ ]:
documents = []

# KPIs
documents.append(f"Latest day revenue is {round(today_rev,2)} and quantity sold is {int(today_qty)}")
documents.append(f"This month revenue is {round(this_month_rev,2)} and growth is {round(month_growth,2)}%")
documents.append(f"This week revenue is {round(this_week_rev,2)} and growth is {round(week_growth,2)}%")

# Top summary
documents.append(f"Top brand is {brand_sales.idxmax()} with revenue {round(brand_sales.max(),2)}")
documents.append(f"Top item is {item_sales.idxmax()} with revenue {round(item_sales.max(),2)}")

# Brand info
for brand, val in brand_sales.head(20).items():
    documents.append(f"Brand {brand} total revenue {round(val,2)}")

# Item info
for item, val in item_sales.head(50).items():
    documents.append(f"Item {item} total revenue {round(val,2)}")

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline

# Embedding model
embed_model = SentenceTransformer('paraphrase-MiniLM-L3-v2')

# LLM - Switched to a compatible text-generation model (distilgpt2)
generator = pipeline("text-generation", model="distilgpt2")

In [ ]:
def get_exact_answer(question):

    q = question.lower()

    numbers = re.findall(r'\\d+', q)
    top_n = int(numbers[0]) if numbers else 5
    top_n = min(top_n, 50)

    # Growth (More flexible checks for growth before general time periods)
    if "growth" in q and "week" in q:
        return f"Week growth is {round(float(week_growth),2)}%"

    if "growth" in q and "month" in q:
        return f"Month growth is {round(float(month_growth),2)}%"

    # Top
    if "top" in q and "revenue" in q and "item" in q:
        return item_sales.head(top_n).to_string()

    if "top" in q and ("sales" in q or "quantity" in q) and "item" in q:
        return df.groupby('Item Name')['Sold Qty'].sum().sort_values(ascending=False).head(top_n).to_string()

    if "top brand" in q:
        return brand_sales.head(top_n).to_string()

    if "top category" in q:
        return category_sales.head(top_n).to_string()

    # Daily
    if "today" in q:
        return f"Today's revenue is {today_rev} and quantity is {today_qty}"

    if "yesterday" in q:
        return f"Yesterday revenue was {yesterday_rev} and quantity was {yesterday_qty}"

    # Week
    if "this week" in q:
        return f"This week revenue is {this_week_rev} and quantity {this_week_qty}"

    if "last week" in q:
        return f"Last week revenue was {last_week_rev} and quantity {last_week_qty}"

    # Month
    if "this month" in q:
        return f"This month revenue is {this_month_rev}"

    if "last month" in q:
        return f"Last month revenue was {last_month_rev}"

    return "No exact data found"

In [ ]:
embeddings = embed_model.encode(documents)

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(np.array(embeddings))

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

# Input box
input_box = widgets.Text(
    placeholder='Ask your business question...',
    description='You:',
    layout=widgets.Layout(width='70%')
)

# Output area
output_area = widgets.Output()

# Button
send_button = widgets.Button(description="Send")

def on_submit(b):
    question = input_box.value

    if question.strip() == "":
        return

    with output_area:
        print(f"\n🧑 You: {question}")

        # Get response
        response = ask_ai(question)

        print(f"🤖 AI: {response}")

    input_box.value = ""  # clear input

send_button.on_click(on_submit)

# Display UI
display(input_box, send_button, output_area)